# CSP Problem - Gruppe C

```Christopher Klein & Colin Vavra```

Für die Prüfungswochen der Hochschule werden *zwei große Prüfungsräume* benötigt. 

*Mehrere Studiengänge* greifen in denselben Wochen auf diese Räume zu. Die verfügbaren *Raum-Zeit-Slots* sollen so verteilt werden, dass die *unterschiedlichen Bedarfe* der Studiengänge berücksichtigt werden und gleichzeitig organisatorisch sinnvolle sowie *faire Belegungspläne* entstehen.

# Variablen + Wertebereiche festlegen

| Slot | Zeitraum    |
| ---- | ----------- |
| A    | 08:00–10:00 |
| B    | 10:30–12:30 |
| C    | 13:00–15:00 |
| D    | 15:30–17:30 |

Zwei Räume
Fünf Tage die Woche

Slots = 5 * 4 * 2 = 40

Studiengaenge = { WIW, ES, ET, INF, MB, MT, DS }

Die Studiengänge *unterscheiden sich in ihrer Größe*. Diese wird durch die *Anzahl der Kursgruppen*
beschrieben, die *in der jeweiligen Prüfungswoche* berücksichtigt werden müssen. Ein Studiengang
mit vielen Kursgruppen besitzt somit einen *höheren Bedarf an Prüfungsraumkapazität* als ein
Studiengang mit wenigen Kursgruppen.

Nicht jeder Studiengang ist in jeder Prüfungswoche beteiligt. Für jede Woche ist im Datensatz angegeben, welche Studiengänge in dieser Woche Prüfungen durchführen. Aus der *Anzahl der Kursgruppen* dieser Studiengänge ergibt sich der *relative Bedarf innerhalb der jeweiligen Woche*.

# Daten

## exam_weeks.csv
- Woche 1-3
- Studiengang
- Anzahl Gruppen

## preferences.json
- Time Slot Preference (1-5)
- Weekday Preference (1-5)
- Structure Preference
  - 3 Preferences
  - Jede enthält
    - condition gegen Anzahl course_groups
    - Parallel vs Sequentiell Preference
    - Preferenzwert + allowed-Boolean

In [3]:
import pandas as pd


# Wochenplan lesen

In [4]:
exam_weeks = pd.read_csv('./data/exam_weeks.csv', delimiter=';')
exam_weeks.columns = ['exam_week','study_program','course_groups','cohorts']
#exam_weeks.head()
print(exam_weeks.to_string())

   exam_week study_program  course_groups  cohorts
0    Woche 1            ES              3        1
1    Woche 1           INF              1        1
2    Woche 1            MB              8        1
3    Woche 1            MT              4        1
4    Woche 1            DS              4        1
5    Woche 2            ET              2        1
6    Woche 2            MB              8        1
7    Woche 2            MT              4        1
8    Woche 2            DS              4        1
9    Woche 3           WIW              5        1
10   Woche 3            ET              2        1
11   Woche 3            MB              8        1
12   Woche 3            DS              4        1


# Modellierung

CSP-basiertes Planungssystem, das für die im Datensatz beschriebenen Prüfungswochen *jeweils* eine *zulässige Verteilung* der *verfügbaren Raum-Zeit-Slots* auf die beteiligten Studiengänge bestimmt.

- Verteilung pro Woche 
- Slot zu Studiengang

```Eine mögliche Modellierung besteht beispielsweise darin, jeder Kombination aus Wochentag, Zeitslot und Raum eine Variable zuzuordnen, deren Wert den zugeordneten Studiengang angibt. Andere fachlich geeignete Repräsentationen sind zulässig.```

A_1 = ( "Mon", "A", "R1" ) 

Zuordnung_1 = (A_1, "MB")



# Lösung

## Schritt 1: Ein CSP pro Woche + Bedarfs-Constraint

**Änderungen gegenüber dem ersten Versuch**
- Die Domäne ist jetzt **wochenspezifisch**: Nur die Studiengänge der jeweiligen Woche sind zulässige Werte.
  → Der Constraint *„In einer Woche dürfen nur die angegebenen Studiengänge Slots erhalten“* ist damit bereits über **D** erfüllt.
- *„Ein Raum kann zu einem Zeitpunkt nur einem Studiengang zugeordnet werden“* ist durch die Modellierung implizit erfüllt: Jede Variable (Tag, Slot, Raum) hat genau **einen** Wert.
- `getSolutions()` ist ungeeignet (bis zu 5⁴⁰ Belegungen). Wir verwenden `getSolution()`.

**Bedarf**

$$q_s = \frac{g_s}{\sum_{j \in W} g_j}, \qquad \lfloor q_s \cdot 40 \rfloor \le n_s \le \lceil q_s \cdot 40 \rceil$$

wobei $n_s$ die Anzahl der Slots ist, die Studiengang $s$ erhält.

In [6]:
from constraint import *

csp = Problem()

weekdays = ["Mo", "Di", "Mi", "Do", "Fr"]
slots = ["A", "B", "C", "D"]
rooms = ["R1", "R2"]

weeks = exam_weeks["exam_week"].unique().tolist()
study_programs = exam_weeks["study_program"].unique().tolist()

# Alle möglichen Kombinationen
domain = [
    (week, day, slot, room)
    for week in weeks
    for day in weekdays
    for slot in slots
    for room in rooms
]

# Eine Variable pro Studiengang
for program in study_programs:
    csp.addVariable(program, domain)

csp.getSolution()   

{'DS': ('Woche 3', 'Fr', 'D', 'R2'),
 'ES': ('Woche 3', 'Fr', 'D', 'R2'),
 'ET': ('Woche 3', 'Fr', 'D', 'R2'),
 'INF': ('Woche 3', 'Fr', 'D', 'R2'),
 'MB': ('Woche 3', 'Fr', 'D', 'R2'),
 'MT': ('Woche 3', 'Fr', 'D', 'R2'),
 'WIW': ('Woche 3', 'Fr', 'D', 'R2')}

In [ ]:
def demand_bounds(week_df):
    groups = dict(zip(
        week_df["study_program"],
        week_df["course_groups"] * week_df["cohorts"]
    ))

    total = sum(groups.values())

    return {
        s: (
            math.floor(g / total * N_SLOTS),
            math.ceil(g / total * N_SLOTS)
        )
        for s, g in groups.items()
    }

Woche 1 {'ES': (6, 6), 'INF': (2, 2), 'MB': (16, 16), 'MT': (8, 8), 'DS': (8, 8)}
Woche 2 {'ET': (4, 5), 'MB': (17, 18), 'MT': (8, 9), 'DS': (8, 9)}
Woche 3 {'WIW': (10, 11), 'ET': (4, 5), 'MB': (16, 17), 'DS': (8, 9)}


In [23]:
def show_plan(solution):
    rows = [{"Tag": v.split("_")[0], "Slot": v.split("_")[1], "Raum": v.split("_")[2], "SG": s}
            for v, s in solution.items()]
    plan = pd.DataFrame(rows).pivot(index=["Slot", "Raum"], columns="Tag", values="SG")
    return plan[weekdays]

solutions = {}
for week in weeks:
    solutions[week] = build_csp(week).getSolution()
    print(week, pd.Series(solutions[week]).value_counts().to_dict())
    display(show_plan(solutions[week]))

Woche 1 {'MB': 16, 'DS': 8, 'MT': 8, 'ES': 6, 'INF': 2}


Tag         Mo  Di  Mi  Do  Fr
Slot Raum                     
A    R1    INF  DS  MB  MT  MB
     R2    INF  DS  MB  MT  MB
B    R1     ES  DS  MB  MT  MB
     R2     ES  DS  MB  MT  MB
C    R1     ES  DS  MB  MT  MB
     R2     ES  DS  MB  MT  MB
D    R1     ES  DS  MB  MT  MB
     R2     ES  DS  MB  MT  MB

Woche 2 {'MB': 18, 'DS': 9, 'MT': 9, 'ET': 4}


Tag        Mo  Di  Mi  Do  Fr
Slot Raum                    
A    R1    MB  DS  MB  DS  MT
     R2    MB  DS  MB  MT  MT
B    R1    MB  DS  MB  MT  MB
     R2    MB  DS  MB  MT  MB
C    R1    ET  DS  MB  MT  MB
     R2    ET  DS  MB  MT  MB
D    R1    ET  DS  MB  MT  MB
     R2    ET  DS  MB  MT  MB

Woche 3 {'MB': 17, 'WIW': 10, 'DS': 9, 'ET': 4}


Tag         Mo  Di   Mi  Do  Fr
Slot Raum                      
A    R1    WIW  DS   MB  DS  MB
     R2    WIW  DS   MB  MB  MB
B    R1    WIW  DS   ET  MB  MB
     R2    WIW  DS   ET  MB  MB
C    R1    WIW  DS   ET  MB  MB
     R2    WIW  DS   ET  MB  MB
D    R1    WIW  DS  WIW  MB  MB
     R2    WIW  DS  WIW  MB  MB